# EDA — `shift.csv`

Вторая таблица. Здесь смены: что, когда, где, за сколько, сколько мест. Это **самая важная таблица для модели** — все парные признаки (`location_match`, `need_mk_match`, ставка/час и т.д.) строятся именно вокруг сменных полей. Плюс именно по `capacity` смены агрегируется итоговая метрика (см. `REGLAMENT.md` п.6).

Что проверяю:
1. Контракт, типы, пропуски, дубли (как и для `user.csv`).
2. Распределения числовых признаков: `hours`, `reward`, `capacity` — особенно интересны выбросы.
3. Временное покрытие: важно для train/val split.
4. Категориальные: `task_type`, `employer_id`, `workplace_id` — концентрация и иерархия.
5. Загадочный признак `id_differential`.
6. Кросс-связь с `user.csv` по `location_id`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 60)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
df = pd.read_csv('/Users/nikolajzuravlev/Documents/Магистратура статьи:книги/Практика/ml-hackathon-main/notebooks/shift.csv')
df['start_at'] = pd.to_datetime(df['start_at'], utc=True)
print(df.shape)
df.head(3)

## 1. Контракт и базовая чистота

In [ ]:
df.info()

In [ ]:
print('NaN:')
print(df.isna().sum())
print()
print(f'Дубликатов строк:      {df.duplicated().sum()}')
print(f'Дубликатов по id:      {df["id"].duplicated().sum()}')
print(f'Уникальных смен:       {df["id"].nunique()}')
print(f'Уникальных локаций:    {df["location_id"].nunique()}')
print(f'Уникальных task_type:  {df["task_type"].nunique()}')
print(f'Уникальных employers:  {df["employer_id"].nunique()}')
print(f'Уникальных workplaces: {df["workplace_id"].nunique()}')

Чисто: 39999 смен, пропусков нет, дубликатов нет, `id` уникальные. По кардинальности уже видна иерархия: 50 работодателей → 1638 точек → 39999 смен. То есть в среднем 24 смены на точку, 800 смен на работодателя. Дальше посмотрю как это распределено.

## 2. Числовые признаки: `hours`, `reward`, `capacity`

In [ ]:
df[['hours', 'reward', 'capacity']].describe().round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

df['hours'].plot.hist(bins=range(1, 24), ax=ax[0], color='#34495e', edgecolor='white')
ax[0].set_title('hours')
ax[0].set_xlabel('часы смены')

df['reward'].plot.hist(bins=40, ax=ax[1], color='#16a085', edgecolor='white')
ax[1].set_title('reward')
ax[1].set_xlabel('руб')

# capacity ограничиваю, иначе из-за выброса 154 ничего не видно
df[df['capacity'] <= 12]['capacity'].plot.hist(bins=range(1, 14), ax=ax[2],
                                               color='#c0392b', edgecolor='white')
ax[2].set_title('capacity (обрезано до 12)')
ax[2].set_xlabel('мест')

plt.tight_layout()

In [ ]:
# Распределение часов — явно две моды (короткие смены ~7ч и длинные ~12ч)
print('hours value_counts:')
print(df['hours'].value_counts().sort_index().to_string())

По `hours` две выраженные моды: 7ч (короткая смена) и 12-13ч (полный рабочий день). Плюс короткий хвост 16-22ч — редкие длинные смены, скорее всего ночные.

По `reward` распределение скошено вправо, медиана 3100, длинный хвост до 9500.

In [ ]:
# capacity — крайне асимметрично
print('capacity value_counts:')
print(df['capacity'].value_counts().sort_index().to_string())
print()
print(f'смен с capacity=1: {(df["capacity"]==1).mean()*100:.1f}%')
print(f'смен с capacity>10: {(df["capacity"]>10).sum()}')

In [ ]:
# 89% смен на 1 место. Глянем что за выброс с capacity=154
outlier = df[df['capacity'] == 154]
outlier[['id', 'start_at', 'location_id', 'task_type', 'hours', 'reward', 'capacity']]

❗ **Выброс по capacity.** Одна смена с `capacity=154` — «Уборка помещений», 12ч, 2200₽. Похоже на разовое мероприятие (генеральная уборка с большой бригадой), не data error. Но для метрики это критично:

Формула из `REGLAMENT.md`: `max_fpr = min(1.0, capacity / 10)`. При capacity ≥ 10 ограничения FPR нет вообще, считается обычный ROC-AUC по TOP-10 кандидатам. То есть смены с `capacity > 10` (всего 7 штук) попадут в собственную capacity-группу при усреднении и будут оцениваться отдельно. Эту смену стоит отслеживать в eval-отчёте — если она попадёт в validation, то будет давать «плавающий» вклад в дневную метрику.

Базовая тенденция: 89% смен — на одно место. То есть основная задача модели — **угадать одного исполнителя из десятка кандидатов**, а не множественный матчинг. Это ставит задачу как **ranking** *(ранжирование — упорядочить кандидатов внутри группы, а не классифицировать каждого отдельно)*, а не классическую бинарную классификацию.

In [ ]:
# Производный признак — ставка ₽/час. Может быть полезен модели как сигнал «выгодная смена / нет»
df['rate_per_hr'] = df['reward'] / df['hours']
print(df['rate_per_hr'].describe().round(1))

## 3. Временной разрез — тут самая большая засада

Смотрю, в какое время идут смены и как они распределены по дням. Это критично, потому что baseline в `training.py` делит train/test 80/20 по времени, а в eval каждый день обрабатывается отдельно.

In [ ]:
print(f'диапазон: {df["start_at"].min().date()} → {df["start_at"].max().date()}')
print(f'уникальных дат: {df["start_at"].dt.date.nunique()}')
print(f'span: {(df["start_at"].max() - df["start_at"].min()).days} дней')

In [ ]:
# Смены в день — ожидаю плюс-минус ровно, но смотрим
per_day = df.groupby(df['start_at'].dt.date).size()

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.bar(per_day.index, per_day.values, color='#2c3e50', width=1.0)
ax.set_title('Число смен по дням')
ax.set_ylabel('shifts')
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.tick_params(axis='x', labelrotation=30)
plt.tight_layout()

Стоп, это не просто сезонность. Видно, что примерно с 8 февраля число смен **резко падает на порядок: с ~1000 в день в среднем (с пиками до 1500) до ~20 в день**. Это не похоже на праздники или выходные. Похоже либо на:
- сэмплинг (организаторы выгрузили основную часть до 8 февраля и редкую часть после),
- ушёл главный заказчик с платформы,
- цензурирование данных (события после какой-то даты вырезаны).

Проверю гипотезу про заказчика. Это **data drift** *(распределение данных меняется во времени — модель, обученная на одном куске, может плохо работать на другом)* — критичная штука для нашей задачи.

In [ ]:
# когда «исчезает» крупнейший работодатель?
top_emp = df['employer_id'].value_counts().index[0]
by_day = df[df['employer_id'] == top_emp].groupby(df['start_at'].dt.date).size()
print(f'Крупнейший employer: {top_emp[:12]}... — {len(df[df["employer_id"]==top_emp])} смен всего')
print(f'Активен в {len(by_day)} из {len(per_day)} дней')
print(f'\nПервые 5 дней:')
print(by_day.head(5).to_string())
print(f'\nПоследние 7 дней присутствия:')
print(by_day.tail(7).to_string())

In [ ]:
# Тоже самое графически: сравню топ-3 работодателей по дням
top3 = df['employer_id'].value_counts().head(3).index.tolist()

fig, ax = plt.subplots(figsize=(13, 4))
for i, emp in enumerate(top3):
    ts = df[df['employer_id'] == emp].groupby(df['start_at'].dt.date).size()
    ax.plot(ts.index, ts.values, label=f'emp {emp[:8]}... ({len(df[df["employer_id"]==emp])} смен)', lw=1.5)

ax.set_title('Активность топ-3 работодателей по дням')
ax.set_ylabel('смен в день')
ax.legend()
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.tick_params(axis='x', labelrotation=30)
plt.tight_layout()

Топ-1 работодатель действительно резко сокращается с 8 февраля. Но это объясняет не весь обвал — проверю, что происходит с остальными:

In [ ]:
# Сравниваю объёмы до и после 8 февраля для всех работодателей
import datetime
border_date = datetime.date(2026, 2, 8)
df['date'] = df['start_at'].dt.date
before = df[df['date'] < border_date]
after = df[df['date'] >= border_date]

print(f'Период до 8 февраля:  {len(before):>5} смен за {before["date"].nunique()} дней (~{len(before)/before["date"].nunique():.0f} смен/день в среднем)')
print(f'Период с 8 февраля:   {len(after):>5} смен за {after["date"].nunique()} дней (~{len(after)/max(1,after["date"].nunique()):.0f} смен/день в среднем)')

# Доля топ-1 работодателя
top_share_before = (before['employer_id']==top_emp).sum() / len(before) * 100
top_share_after = (after['employer_id']==top_emp).sum() / max(1,len(after)) * 100
print(f'\nДоля топ-1 работодателя (fb3d1aaa...):')
print(f'  до 8 февр: {(before["employer_id"]==top_emp).sum()} смен ({top_share_before:.1f}% периода)')
print(f'  с 8 февр:  {(after["employer_id"]==top_emp).sum()} смен ({top_share_after:.1f}% периода)')

# Если убрать топ-1 — что остаётся?
without_top_before = before[before['employer_id']!=top_emp]
without_top_after = after[after['employer_id']!=top_emp]
rate_before = len(without_top_before) / before['date'].nunique()
rate_after = len(without_top_after) / max(1, after['date'].nunique())
print(f'\nЕсли исключить топ-1:')
print(f'  до 8 февр: {rate_before:.0f} смен/день в среднем')
print(f'  с 8 февр:  {rate_after:.0f} смен/день в среднем')
print(f'  → падение в {rate_before/max(1,rate_after):.0f}x даже без топ-1')

**Уточнение причины drift.** Картина сложнее, чем просто «ушёл крупнейший работодатель»:
- Топ-1 работодатель действительно почти ушёл (с 12598 → 27 смен). Это даёт очень большой вклад.
- Но **остальные тоже схлопнулись**: без топ-1, среднее падает с ~700 смен/день до ~20. То есть платформа в целом сократилась на порядок, не только из-за одного работодателя.

Возможные объяснения:
- Это сэмплинг данных (организаторы выгрузили плотный кусок + редкий хвост для проверки экстремальных условий).
- Это обрезание датасета по какой-то дате (после 8 февраля попало то, что попалось).
- Это реальный отток платформы — но синхронный отток у всех работодателей сразу маловероятен в реальном бизнесе.

Для нас важно практически: распределение в проде может оказаться любым из этих режимов, и **модель должна уметь работать на разрежённом периоде**, а не только на плотном январе. Стоит спросить организаторов: «какой режим в `validation/`?».

### Симуляция baseline-сплита 80/20

В `training.py:_time_split` разделение идёт **по уникальным таймстемпам**, не по строкам. Из-за обвала после 8 февраля это даёт совсем не 80/20 по строкам. Считаем.

In [ ]:
# воспроизвожу логику _time_split один-в-один
unique_ts = np.array(sorted(df['start_at'].dropna().unique()))
split_idx = max(1, int(unique_ts.size * (1 - 0.2)))
split_idx = min(split_idx, unique_ts.size - 1)
border = unique_ts[split_idx]

train_part = df[df['start_at'] < border]
test_part = df[df['start_at'] >= border]

print(f'граница: {border}')
print(f'train rows: {len(train_part):>6} ({len(train_part)/len(df)*100:.1f}% — а должно быть ~80%)')
print(f'test rows:  {len(test_part):>6} ({len(test_part)/len(df)*100:.1f}% — а должно быть ~20%)')
print(f'\ntrain даты:  {train_part["start_at"].dt.date.nunique():>3} ({train_part["start_at"].min().date()} → {train_part["start_at"].max().date()})')
print(f'test даты:   {test_part["start_at"].dt.date.nunique():>3} ({test_part["start_at"].min().date()} → {test_part["start_at"].max().date()})')

❗ **Большой риск для baseline.** Сплит 80/20 по таймстемпам даёт **~96/4 по строкам**. Test-выборка получается всего ~1700 строк — на ней метрика оценивается с очень высокой дисперсией, и не отражает реальность.

**Почему так получилось.** Чтобы делать честный time-based split (модель не должна учиться на будущем), `_time_split` берёт все уникальные значения `start_at`, отсчитывает 80% от их количества и режет по этой границе. Логика разумная, но в наших данных в феврале объём смен в день упал в ~50 раз. Уникальные таймстемпы остались (просто их стало мало), а строк за каждым из них — почти ничего. Поэтому формальные «20% таймстемпов» оказываются «4% строк».

**Почему это плохо.** На test ~1700 строк ROC-AUC будет «прыгать» от запуска к запуску в зависимости от случайностей. Команда увидит цифру и подумает что модель улучшилась/ухудшилась, а это просто шум. Плюс test покрывает только разрежённый февраль — мы не проверяем, как модель работает на «нормальном» распределении января.

**Что предложить команде — три варианта, от простого к надёжному:**

**Вариант 1 (минимальная правка): сплит по доле строк.**
Меняем одну строку в `_time_split`: вместо percentile по `unique_ts` берём percentile по `len(df)`. То есть — сортируем по `start_at`, берём первые 80% строк в train, последние 20% в test. Граница попадёт куда-то в начале февраля, и обе выборки получатся плотные.
Минус: test всё равно будет в основном состоять из «начала февраля», где распределение уже начало меняться. Но это сильно лучше текущих 4%.

**Вариант 2 (надёжнее): time-series cross-validation.**
Не один сплит, а несколько с скользящим окном. Например, по неделям:
- fold 1: train = неделя 1, val = неделя 2
- fold 2: train = недели 1-2, val = неделя 3
- fold 3: train = недели 1-3, val = неделя 4
- ...

Это называется **expanding window CV** *(расширяющееся окно — train постепенно растёт)*. Альтернатива — **rolling window** *(скользящее окно — фиксированный размер train, например, всегда «последние 2 недели»)*. Метрика — среднее по фолдам, и видно стабильность.
Минус: дольше считать (несколько train-циклов вместо одного), сложнее отлаживать.
В sklearn есть готовый `TimeSeriesSplit`, можно взять его.

**Вариант 3 (самый правильный, но требует решения от команды): два режима.**
Учитывая что распределение в феврале — это другой режим работы платформы (см. раздел про data drift), можно делать две модели или одну, но с целевым выбором train-периода:
- если организаторы скажут что `validation/` — это плотный период (январь), учим на январе;
- если разрежённый (после 8 февраля) — учим на хвосте, иначе модель будет переобучаться на «доминирующем работодателе», которого в validation нет.

**Что обязательно сделать в любом варианте:**
- Замерить размеры train и test в строках после изменения сплита (а не в днях/таймстемпах).
- Сравнить распределения в train и test по ключевым полям: `task_type`, `employer_id`, target rate. Если они радикально разные — сплит всё ещё битый.
- Проверить что в test есть смены хотя бы из нескольких разных локаций и task_type, иначе ROC-AUC посчитается на нерепрезентативной выборке.

In [ ]:
# Часы старта смены — для понимания паттерна
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_hour = df['start_at'].dt.hour.value_counts().sort_index()
by_weekday = df['start_at'].dt.day_name().value_counts().reindex(weekday_order)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(by_hour.index, by_hour.values, color='#34495e')
ax[0].set_title('Час старта смены')
ax[0].set_xlabel('час'); ax[0].set_xticks(range(0, 24, 2))

ax[1].bar(range(7), by_weekday.values, color='#16a085')
ax[1].set_xticks(range(7)); ax[1].set_xticklabels([d[:3] for d in weekday_order])
ax[1].set_title('Смены по дням недели')
plt.tight_layout()

Жёсткий пик в 8-9 утра (стандартный старт рабочей смены). Дни недели почти ровно, лёгкий перевес на пт/сб (выходные торговля).

## 4. Категориальные признаки

### `task_type` — что вообще делают исполнители

In [ ]:
task_counts = df['task_type'].value_counts()
print(f'Уникальных task_type: {len(task_counts)}')
print(f'\nТоп-10:')
print(task_counts.head(10).to_string())
print(f'\nХвост (5 самых редких):')
print(task_counts.tail(5).to_string())
print(f'\nДоля топ-5: {task_counts.head(5).sum() / len(df) * 100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
task_counts.head(15).plot.barh(ax=ax, color='#2980b9')
ax.invert_yaxis()
ax.set_xlabel('кол-во смен')
ax.set_title('Топ-15 task_type по числу смен')
plt.tight_layout()

27 типов задач, доминирует «Выкладка товара» (43% всех смен). Это ритейл-платформа.

Хвост из 5-7 редких задач — модели на них учиться не на чем. В feature engineering имеет смысл:
- топ-10 типов оставить как есть
- хвост схлопнуть в `task_type_other` (бакетизация — для линейных моделей)
- для CatBoost можно скармливать как есть

### `task_type` × `need_mk` — сильнейший доменный сигнал

Гипотеза: для еды нужна медкнижка, для разгрузки — нет. Если корреляция чёткая — это означает, что `need_mk` — почти функция от `task_type` (мультиколлинеарность *= когда два признака практически дублируют друг друга, и линейная модель не может развести их вклады*), и для модели они дублируются.

In [ ]:
mk_by_task = (
    df.groupby('task_type')
      .agg(n=('id', 'size'), need_mk_rate=('need_mk', 'mean'))
      .query('n >= 50')
      .sort_values('need_mk_rate')
)
mk_by_task.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
y = np.arange(len(mk_by_task))
colors = ['#27ae60' if v == 1 else '#c0392b' if v == 0 else '#f39c12' for v in mk_by_task['need_mk_rate']]
ax.barh(y, mk_by_task['need_mk_rate'].values, color=colors)
ax.set_yticks(y); ax.set_yticklabels(mk_by_task.index, fontsize=9)
ax.set_xlabel('доля смен с need_mk = True')
ax.invert_yaxis()
ax.set_title('need_mk по task_type (только task_type с n>=50)')
plt.tight_layout()

Картина чёткая и доменно-логичная:
- **0% need_mk**: выдача заказов OZON/Wildberries, доставка пешком, уборка на улице.
- **100% need_mk**: всё что связано с едой (приготовление, мойка посуды, прикассовая зона, обслуживание покупателей).
- **Промежуточные** (53-94%): доставка на авто/велосипеде, погрузка, уборка помещений, выкладка — где политика зависит от конкретного работодателя.

Для модели:
- `need_mk` — почти детерминированная функция от `task_type` для крайних категорий. Линейная регрессия эту мультиколлинеарность переварит, но Random Forest/CatBoost будет проще обучить, если оставить оба.
- Парный признак `need_mk_match = (has_mk == need_mk)` (уже в baseline) — это **жёсткий допуск**, без которого юзер физически не может работать на смене. Будет одной из топ-фич.

### `employer_id` и `workplace_id` — концентрация и иерархия

In [ ]:
emp_counts = df['employer_id'].value_counts()
print(f'Уникальных работодателей: {len(emp_counts)}')
print(f'\nТоп-5 (доля от всех смен):')
for i, (emp, n) in enumerate(emp_counts.head(5).items(), 1):
    print(f'  {i}. {emp[:12]}...  {n:>5}  ({n/len(df)*100:.1f}%)')
print(f'\nТоп-5 совокупно: {emp_counts.head(5).sum() / len(df) * 100:.1f}% всех смен')
print(f'Топ-1 в одиночку: {emp_counts.iloc[0] / len(df) * 100:.1f}%')

In [ ]:
# каждый workplace принадлежит одному employer? (важно для feature engineering)
wp_per_emp = df.groupby('workplace_id')['employer_id'].nunique()
print(f'workplaces, привязанных к нескольким employers: {(wp_per_emp > 1).sum()}')
print(f'значит workplace_id — строгая подкатегория employer_id')

Структура такая: **`employer_id` ⊃ `workplace_id`** — иерархическая. Каждая точка работы привязана ровно к одному работодателю. Это упрощает feature engineering: можно использовать любой из двух уровней без дублирующего сигнала.

Концентрация чудовищная: 5 работодателей дают 82% смен, один — 32%. Если этот один работодатель «исчез» во второй половине датасета (что мы видели выше) — модель будет очень чувствительна к смене распределения.

## 5. Загадочный `id_differential`

Поле есть в контракте (булево), но что физически означает — нигде не объяснено. У 3.6% смен `True`. Попробую понять по корреляциям, с чем оно связано.

In [ ]:
print(f'id_differential = True:  {df["id_differential"].sum()} ({df["id_differential"].mean()*100:.1f}%)')
print(f'id_differential = False: {(~df["id_differential"]).sum()}')
print()
print('Базовые сравнения групп:')
comparison = df.groupby('id_differential').agg(
    n=('id', 'size'),
    median_reward=('reward', 'median'),
    median_rate=('rate_per_hr', 'median'),
    median_hours=('hours', 'median'),
    need_mk_rate=('need_mk', 'mean')
).round(2)
print(comparison)

In [ ]:
# Разрез по task_type — где `id_differential=True` встречается?
diff_by_task = (
    df.groupby('task_type')
      .agg(n=('id', 'size'), diff_rate=('id_differential', 'mean'))
      .query('n >= 50')
      .sort_values('diff_rate', ascending=False)
      .round(2)
)
diff_by_task

❗ **Загадка решена.** Для всех task_type `diff_rate` либо ровно 0, либо близко к 1 — то есть это бинарный маркер типа задачи, а не независимый признак:
- 100% «Доставка заказов на велосипеде»
- 83% «Доставка заказов на авто»
- 0% всё остальное

Иными словами, **`id_differential=True` ≡ «это доставочная вакансия»**. Скорее всего расшифровывается как «differential ID» — другие требования к документам (права, самозанятость, ИНН и т.д., курьеры — отдельная категория исполнителей с другими экономическими условиями).

Это объясняет и разницу в зарплатах: `id_differential=True` платит в среднем в 2 раза больше (median 5500₽ vs 3000₽) — потому что курьерство традиционно дороже за час из-за расходов на транспорт.

Для модели:
- Признак **избыточен с `task_type`** — если task_type = «Доставка...», то `id_differential` тоже True. Линейные модели и бустинги выведут эту связь сами.
- Но он **полезен как простой бинарный флаг «доставка vs не-доставка»** — особенно если бакетизировать `task_type` (объединять «Доставка на авто» + «Доставка на велосипеде» теряют гранулярность). Тогда `id_differential` восстанавливает этот сигнал.
- Для **сервиса** имеет значение: к `id_differential=True` смене кандидаты должны быть с правами/велосипедом, и таких юзеров платформа должна как-то отличать. У нас в `user.csv` такой колонки нет — это может быть ещё один cold-start кейс.

## 6. Кросс-связь с `user.csv` по `location_id`

Это важно: модель ищет матч `user → shift`. Если у юзера локация, в которой нет смен — он бесполезен в датасете. Если у смены локация, в которой нет юзеров — задача неразрешима без cold-start логики *(cold-start = ситуация когда у объекта нет истории, новый юзер или новая смена, и модели нечего на нём считать)*.

In [ ]:
users = pd.read_csv('/Users/nikolajzuravlev/Documents/Магистратура статьи:книги/Практика/ml-hackathon-main/notebooks/user.csv')
user_locs = set(users['location_id'].unique())
shift_locs = set(df['location_id'].unique())

both = user_locs & shift_locs
shift_only = shift_locs - user_locs
user_only = user_locs - shift_locs

print(f'локаций в обеих таблицах:    {len(both):>4}')
print(f'локаций только в shifts:     {len(shift_only):>4}  ← в этих локациях нет ни одного зарегистрированного юзера')
print(f'локаций только в users:      {len(user_only):>4}  ← в этих локациях нет смен, юзеры неактивны')
print()
shift_only_count = df[df['location_id'].isin(shift_only)].shape[0]
user_only_count = users[users['location_id'].isin(user_only)].shape[0]
print(f'смен в "orphan" локациях:    {shift_only_count:>5}  ({shift_only_count/len(df)*100:.1f}% всех смен)')
print(f'юзеров в "orphan" локациях:  {user_only_count:>5}  ({user_only_count/len(users)*100:.1f}% всех юзеров)')

**Что увидели в цифрах.** Подавляющее большинство смен (~99%) и юзеров (~99%) находятся в локациях, где есть и те, и другие — то есть для основной массы данных матчинг возможен в принципе. «Сиротские» локации (только смены без юзеров, или только юзеры без смен) суммарно дают несколько процентов и общую модель не сломают.

**Но эти несколько процентов всё равно требуют отдельной обработки** — иначе на проде в этих случаях сервис либо отдаст пустой ответ, либо упадёт с ошибкой. Разберём оба кейса:

**Кейс 1: смены в локациях без юзеров (~40 локаций, ~1% смен).**
Это смены в районах, где у платформы вообще нет зарегистрированных исполнителей. Модель ранжирования здесь беспомощна — некого ранжировать. На стороне сервиса нужно решить, что возвращать в этом случае:
- *Самое простое:* возвращать пустой ответ + флаг «нет кандидатов в локации». Дальше работодатель видит «никого не нашлось» и принимает решение сам.
- *Получше:* расширять зону поиска на соседние локации. Но для этого нужна **геоинформация** — координаты или хотя бы матрица «какая локация рядом с какой». В наших данных её нет, есть только id. Можно либо запросить такую матрицу у организаторов, либо построить эвристику (например, по близости номеров id, если они структурированы).
- *Альтернатива:* возвращать топ-N юзеров из соседнего региона / всей платформы как «backup», но с пометкой «не в локации». Это усложняет логику и может ухудшить метрику если organizers строго считают по локации.

Что нужно от команды: договориться какой из вариантов реализуем, и кто это будет делать на стороне сервиса. Это **не задача модели** — это задача candidate generation *(отбора кандидатов до ранжирования)*.

**Кейс 2: юзеры в локациях без смен (~1% юзеров).**
Эти юзеры зарегистрировались в локациях, где работодатели вообще не выкладывают смены. Они физически не могли получить релевантное предложение — поэтому в `event.csv` у них либо ноль активности (если они не смотрели смены вне своей локации), либо очень мало (если случайно посмотрели что-то). Это автоматически делает их «холодными» для нашей модели.

Полезно использовать их как **тестовый набор для оценки cold-start качества модели**: сначала оценить их моделью, потом посмотреть какие скоры она им даёт. Если она систематически даёт им низкий скор — значит, ловит cold-start правильно. Если даёт случайно — значит, в фичах не хватает признаков типа `is_cold_user` (см. сводку), и модель путает «холодных» с «активными».

## 7. Чек-сводка для команды

**Качество данных**
- 39999 смен, 11 колонок. Пропусков и дубликатов нет, контракт соблюдён, типы корректные.

**Что важно знать перед разработкой модели**

1. **Data drift во времени.** Данные неоднородны: до ~7 февраля идёт ~1000 смен в день в среднем (с пиками до 1500), после — обвал до ~20 в день. Уход топ-1 работодателя (32% всех смен) объясняет существенную часть обвала, но не всю: остальные работодатели тоже сокращаются на порядок. Это не баг датасета, это реальное смещение распределения.

2. **Baseline-сплит сломан.** Текущий `_time_split(test_ratio=0.2)` режет 80/20 по уникальным таймстемпам, но из-за разрежённости февраля по строкам это даёт ~96/4. Test получается всего ~1700 строк — на такой выборке ROC-AUC скачет от запуска к запуску из-за случайностей, а не из-за качества модели. Команда будет смотреть на нерепрезентативную метрику. Подробное обсуждение и три варианта починки — в разделе «Симуляция baseline-сплита 80/20» выше; короткая версия — менять либо на percentile по строкам, либо на TimeSeriesSplit с несколькими фолдами.

3. **89% смен — на одно место** (`capacity=1`). На уровне постановки задачи это значит, что модель в основном решает «выбери одного лучшего из всех кандидатов в локации», а не «выбери N подходящих». Для оставшихся 11% смен с `capacity > 1` логика «выбери N» тоже работает, но они редкие — оптимизировать модель под них специально не нужно, главное чтобы не ломалась.

4. **Один выброс `capacity = 154`** (генеральная уборка). Это не data error — реальная смена с массовым набором людей. В метрике организаторов смены группируются по `capacity` и усредняются внутри группы; этот outlier попадёт в свою отдельную группу и будет давать «прыгающий» вклад в дневную метрику. Не баг, но при анализе результатов eval надо помнить про неё, иначе можно подумать что модель «нестабильная», хотя на самом деле это одна аномальная смена сдвигает агрегаты.

5. **Концентрация работодателей**: 5 работодателей дают 82% смен, один — 32%. Это значит, что модель будет в основном «выучивать» паттерны этих 5 работодателей, и если в `validation/` доминируют другие — качество просядет. Связано с пунктом 1 (drift): доминирующий работодатель почти ушёл во второй половине датасета, и если организаторы будут проверять модель на февральских данных — её надо учить на февральских же.

6. **`workplace_id` строго вложен в `employer_id`** — иерархическая категория. Каждая `workplace_id` (точка работы — конкретный магазин или склад) принадлежит ровно одному `employer_id` (работодатель — компания, которая выкладывает смены). Использовать в фичах одновременно оба уровня бессмысленно: один полностью определяет другой, для линейной модели это будет мультиколлинеарность *(линейная зависимость двух признаков, ломает интерпретацию весов)*. Решение: брать **один уровень** по выбору — `employer_id` если хочется грубее (50 категорий), `workplace_id` если детальнее (1638 категорий). Для бустингов это менее критично, но всё равно лучше дать один.

7. **`id_differential` совпадает с доставочными task_type** (100% совпадение в train). В feature engineering можно или не брать вовсе (избыточно с `task_type`), или использовать как удобный бинарный флаг «доставка vs не-доставка», особенно если делаем бакетизацию `task_type` и сливаем разные типы доставки в одну категорию. См. также `EDA_SUMMARY.md`.

**Что предлагаю в feature engineering** (полная приоритизация в `EDA_SUMMARY.md`):

- **`rate_per_hr = reward / hours`** — ставка в рублях за час. Это **топ-приоритет**: смены с высокой ставкой за час получают больше просмотров и откликов (мы это видели в `eda_event.ipynb`). Без этого признака модель видит `reward` и `hours` отдельно, и линейная модель не сможет их соотнести в правильной пропорции — нужна явная производная.

- **`start_hour`, `start_weekday`** — час и день недели старта смены. Из `start_at` извлекаем компоненты времени. Зачем: распределение спроса на работу зависит от времени — утренние смены (8-9 утра) популярнее ночных, выходные смены имеют другой профиль кандидатов. Модель из числового `start_at` сама эти паттерны не выудит, нужно вынуть руками. Категориальные признаки 0-23 (час) и 0-6 (день).

- **`is_long_shift = hours >= 12`, `is_high_capacity = capacity >= 3`** — простые бинарные пороги. Длинные смены (12+ часов) — это другой тип работы (полный рабочий день, чаще ночные/посуточные), и привлекают других людей. Смены с большой capacity — это не один-в-один матчинг, а массовый набор, тоже другая логика. Эти бинарные флаги дают модели «переключатель режима» вместо размытого числового сигнала.

- **Бакетизация хвоста `task_type`** *(объединение редких категорий в одну)*. У нас 27 типов задач, но 5-7 из них встречаются меньше 50 раз. Если оставить их как отдельные категории, линейная модель попытается выучить вес под каждую и переобучится на 1-5 примерах. Решение: всё с `n < 50` склеиваем в категорию `task_type_other`. Для бустингов не нужно — они сами регуляризуют редкие категории.

- **`task_type_freq`** — frequency encoding *(замена id категории на её частоту в train)*. Для каждого `task_type` записываем сколько раз он встречается. «Выкладка товара» → 17000+, какая-нибудь редкая категория → 30. Это даёт линейной модели одно числовое поле вместо 27 one-hot колонок и одновременно несёт сигнал «насколько эта задача типична». Для бустингов опять же не обязательно.

**Открытые вопросы (см. `EDA_SUMMARY.md` для полного списка)**

- **Какой период покрывает `validation/shift.csv`?** Это самый практически важный вопрос. Если validation — это плотный период (январь), модель надо учить на январе. Если разрежённый февраль — учить на февральском хвосте, иначе модель переобучится на доминирующего работодателя, которого в проде не будет. От ответа зависит на каких данных тренироваться, и неправильный выбор — это сразу -10-20% качества.

- **Что делать в сервисе на `predict`, если у смены `location_id` без зарегистрированных юзеров?** Это ~1% смен. Модель ранжирования на пустом множестве кандидатов работать не может. Команде нужно решить — возвращать пустой ответ, расширять поиск на соседние локации, или fallback на топ-юзеров платформы. Это решение про сервис, а не про модель.

- **Что физически означает `id_differential`?** В наших данных 100% совпадает с доставочными task_type, но это может быть случайным совпадением одной выборки. Стоит подтвердить у организаторов — это даст уверенность, можно ли его использовать как стабильный флаг или это артефакт.